# 04 Baselines: simple references (E09)

**Purpose.** Score the validation set with simple, untrained rankings, each sorting the candidates by one signal. They are the reference points every trained model is compared with, and they show how strong popularity, relevance and personal history are on their own. Team 3's model and its correction ladder follow in a later part of this notebook.

**Which references** (fixed in advance from training-row evidence, not chosen on validation):

| Reference | Feature | Why |
|---|---|---|
| Random order | constant score | floor (E02) |
| BM25 | `bm25` | standard lexical relevance |
| Query-title Jaccard | `title_jaccard` | best lexical signal on training rows (E07) |
| Training exposures | `pop_exposures` | strongest non-history popularity on training rows (E07, E08) |
| History popularity | `pop_history` | strongest single signal on training rows (E08) |
| Engagement rate | `pop_engaged_rate` | supervised popularity |
| Same product, all history | `pooled_product` | best personal signal (E03) |
| Same shop, other products | `pooled_shop_other` | personal evidence that transfers to new products (E05b) |
| Same product, recent clicks | `click_0_7_product` | strongest window block (E03) |
| Same product, query-related history | `qrel_product` | query relevance of history (Q1) |

**Rules.**
- Validation searches with at least one engaged candidate (19,772), complete lists, exact expectation over random tie order (`scripts/evaluation.py`). Many candidates tie (e.g. zero popularity); ties are never broken by file order.
- Missing values (`NaN`, unknown metadata) are scored as 0: unknown counts as "no evidence", neither better nor worse than no match.
- Nothing is tuned here. Each result goes into `docs/experiment_results.csv` as experiment E09.

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_ROOT = next((p.resolve() for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "scripts/evaluation.py").is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Open/clone the repository checkout first.")
if IN_COLAB:
    drive.mount("/content/drive")
PROCESSED = REPO_ROOT / "data/processed"

sys.path.insert(0, str(REPO_ROOT))
from scripts.evaluation import RESULTS, evaluate, log_result, paired_difference, summarise

pd.options.display.float_format = "{:,.4f}".format
FEATURES = "v3"  # E09 ran on v2; the reference features are identical in v3
meta = json.loads((PROCESSED / f"features_{FEATURES}_meta.json").read_text())
searches = pd.read_parquet(PROCESSED / "search_table.parquet")
assignment = pd.read_csv(PROCESSED / "split_v1.csv", index_col="session_id")
validation_ids = assignment.index[assignment.split == "validation"]
validation_sizes = searches.loc[validation_ids, "n_candidates"]

## 1. Load the validation features

In [ ]:
REFERENCES = {
    "random order": None,
    "BM25": "bm25",
    "query-title Jaccard": "title_jaccard",
    "training exposures": "pop_exposures",
    "history popularity": "pop_history",
    "history popularity, lag 1 day": "pop_history_lag1d",
    "engagement rate": "pop_engaged_rate",
    "same product, all history": "pooled_product",
    "same shop, other products": "pooled_shop_other",
    "same product, recent clicks": "click_0_7_product",
    "same product, query-related history": "qrel_product",
}
columns = ["session_id", "label"] + [f for f in REFERENCES.values() if f]
validation = pd.read_parquet(PROCESSED / f"features_{FEATURES}_validation.parquet", columns=columns)
print(f"validation: {validation.session_id.nunique():,} searches, {len(validation):,} candidates")
validation[[f for f in REFERENCES.values() if f]].isna().mean().rename("missing share").to_frame()

## 2. Evaluate every reference

In [ ]:
per_search = {}
for name, feature in REFERENCES.items():
    validation["score"] = 0.0 if feature is None else validation[feature].fillna(0).astype(float)
    per_search[name] = evaluate(validation, "score", expected_sizes=validation_sizes)

results = pd.concat({name: summarise(per).iloc[0] for name, per in per_search.items()}, axis=1).T
results["searches"] = results.searches.astype(int)
results = results.sort_values("ndcg_at_10", ascending=False)
results

Paired differences on the same searches (mean and 95% bootstrap interval of the per-search difference): against random order, and against history popularity, the strongest non-personal reference on training rows.

In [ ]:
rows = {}
for name in results.index:
    vs_random = paired_difference(per_search[name], per_search["random order"])
    vs_popularity = paired_difference(per_search[name], per_search["history popularity"])
    rows[name] = {"vs_random": vs_random[0], "vs_random_low": vs_random[1], "vs_random_high": vs_random[2],
                  "vs_history_pop": vs_popularity[0], "vs_history_pop_low": vs_popularity[1],
                  "vs_history_pop_high": vs_popularity[2]}
differences = pd.DataFrame(rows).T
differences

## 3. Log to the results registry

One row per reference (experiment E09). Re-running the notebook does not duplicate rows.

In [ ]:
logged = pd.read_csv(RESULTS) if RESULTS.exists() else pd.DataFrame(columns=["experiment", "model"])
for name, feature in REFERENCES.items():
    if name == "random order" or ((logged.experiment == "E09") & (logged.model == name)).any():
        continue  # random order is already logged as E02
    experiment = "E13" if feature == "pop_history_lag1d" else "E09"  # the lagged reference was added in E13
    if ((logged.experiment == experiment) & (logged.model == name) & (logged.split == "validation")).any():
        continue
    log_result(experiment, name, "validation", per_search[name],
               notes=f"untrained: sort by {feature} (features {FEATURES}); NaN as 0")
pd.read_csv(RESULTS)

## 4. By segment

The random-order floor differs between segments (E02), so each segment is read as the gain over random **within** that segment.

In [ ]:
info = searches.loc[per_search["random order"].index]
segments = {
    "activity in last 7 days": np.where(info.days_since_last_action < 7, "yes", "no"),
    "list size": pd.cut(info.n_candidates, [0, 30, 90, 150, 200], labels=["1-30", "31-90", "91-150", "151-200"]),
    "history length": pd.cut(info.history_len, [0, 30, 100, 300, np.inf], labels=["1-30", "31-100", "101-300", "300+"]),
}
shown = ["history popularity", "training exposures", "same product, all history", "same product, recent clicks",
         "query-title Jaccard"]
tables = {}
for segment, labels in segments.items():
    by = pd.Series(labels, index=info.index)
    random_mean = per_search["random order"].ndcg.groupby(by, observed=True).mean()
    table = pd.DataFrame({name: per_search[name].ndcg.groupby(by, observed=True).mean() - random_mean for name in shown})
    table.insert(0, "searches", by.value_counts())
    table.insert(1, "random", random_mean)
    tables[segment] = table
pd.concat(tables)

## 5. Clean population (E13)

The data-generation audit (E12) found that 5.3% of searches have a final gap clamped to exactly 1 second: their histories most likely contain actions from during or after the final search. From E13 on, the main evaluation uses the remaining ("clean") searches; all searches are the sensitivity check. Here every reference is reported on both, and the clean results are logged as E13 (split `validation_clean`).

In [ ]:
clean = ~searches.final_gap_clamped
clean_ids = per_search["random order"].index[clean.reindex(per_search["random order"].index).to_numpy()]
print(f"validation searches with an engaged candidate: {len(per_search['random order']):,}; "
      f"clean: {len(clean_ids):,}; clamped: {len(per_search['random order']) - len(clean_ids):,}")
both = pd.DataFrame({
    "all searches": {n: per_search[n].ndcg.mean() for n in REFERENCES},
    "clean searches": {n: per_search[n].ndcg.loc[clean_ids].mean() for n in REFERENCES},
    "clamped searches": {n: per_search[n].ndcg.drop(clean_ids).mean() for n in REFERENCES},
})
both["clean minus all"] = both["clean searches"] - both["all searches"]
logged = pd.read_csv(RESULTS)
for name, feature in REFERENCES.items():
    if ((logged.experiment == "E13") & (logged.model == name) & (logged.split == "validation_clean")).any():
        continue
    log_result("E13", name, "validation_clean", per_search[name].loc[clean_ids],
               notes=f"clean population (final gap not clamped); sort by {feature or 'constant'} (features {FEATURES})")
both.sort_values("clean searches", ascending=False)

## Summary of the simple references (E09)

- **Popularity dominates the simple references on validation:** history popularity reaches NDCG@10 0.341 (+0.257 over random), training exposures 0.315, the engagement rate 0.245. The order and sizes match the training-row results (E07, E08), so the signal is not an artefact of out-of-fold encoding.
- **Best personal reference:** same product in all history, 0.182 (+0.098), i.e. 0.159 below history popularity [0.153, 0.164]. Recent clicks alone 0.155; shop via other products 0.136; query-related history 0.136.
- **Lexical relevance alone barely beats random:** Jaccard 0.097, BM25 0.093.
- **Segments:** popularity helps equally with and without activity in the last 7 days (+0.256 / +0.257), personal history mostly with recent activity (+0.107 vs +0.027) and more with longer histories (+0.076 for up to 30 actions, +0.118 above 300). Popularity gains are largest for lists of 31-90 candidates (+0.326).
- **Consequence:** the trained non-personalised ranker (G0-G2) is the reference that personalisation must beat. Whether personal history adds on top of popularity is the question for `05_models`; single signals cannot answer it.

# Team 3, part 1: their evaluation protocol (E10)

Team 3 (the prior baseline every group received; diagnosis in FINDINGS 5) reported **HR@10 0.978 and NDCG@10 0.568** for their hybrid NCF + LSTM model. Their evaluation differs from ours in three ways:

1. **Downsampled lists** (`Data Preparation/1. Sampling.ipynb`): every engaged candidate plus `min(4 x positives, negatives)` randomly drawn non-engaged ones; lists shrink from ~89 to ~9 candidates.
2. **Exponential gains:** `(2^label - 1) / log2(rank + 1)`, so a purchase counts 7, a cart 3, a click 1 (we use 3 / 2 / 1).
3. **Tie handling:** `np.argsort(scores)[::-1]`, ties broken by array order, which in their pipeline is engaged-first.

Here we apply their protocol to **our** validation searches, without any model, to see what it does to rankings whose quality we know: random order, history popularity and same-product history. Their NDCG function is copied verbatim (`Models/3.1_Hybrid_NCF_with_LSTM.ipynb`, cell 91). Random draws: 20 random orders per search (ties broken at random, never by array order), seeded per search.

In [ ]:
def team3_ndcg_at_k(user_preds, user_labels, K=10):
    # Verbatim from Team 3 (Models/3.1_Hybrid_NCF_with_LSTM.ipynb, cell 91).
    topk_indices = np.argsort(user_preds)[::-1][:K]
    topk_labels = np.array(user_labels)[topk_indices]
    dcg = np.sum((2**topk_labels - 1) / np.log2(np.arange(1, len(topk_labels)+1) + 1))
    ideal_labels = np.sort(user_labels)[::-1][:K]
    idcg = np.sum((2**ideal_labels - 1) / np.log2(np.arange(1, len(ideal_labels)+1) + 1))
    return dcg / idcg if idcg > 0 else 0.0


def team3_hit_rate_at_k(user_preds, user_labels, K=10):
    # Verbatim from Team 3 (cell 90).
    topk_indices = np.argsort(user_preds)[::-1][:K]
    topk_labels = np.array(user_labels)[topk_indices]
    return int((topk_labels > 0).any())


def team3_sample(labels, rng):
    # Their sampling: all positives + min(4 x positives, negatives) random negatives.
    pos, neg = np.flatnonzero(labels > 0), np.flatnonzero(labels == 0)
    k = min(4 * len(pos), len(neg))
    return np.concatenate([pos, rng.choice(neg, size=k, replace=False)]) if k else pos


def team3_scores(scores, labels, draws, seed):
    # Mean Team 3 NDCG@10 and HR@10 over random tie orders (scores permuted, then a stable descending sort).
    rng = np.random.default_rng(seed)
    ndcg = hit = 0.0
    for _ in range(draws):
        perm = rng.permutation(len(scores))
        s, l = scores[perm], labels[perm]
        jitter = rng.random(len(s)) * 1e-9 * (np.ptp(s) + 1)  # breaks ties at random, keeps the order otherwise
        ndcg += team3_ndcg_at_k(s + jitter, l)
        hit += team3_hit_rate_at_k(s + jitter, l)
    return ndcg / draws, hit / draws


assert team3_ndcg_at_k(np.array([2.0, 1.0]), np.array([3, 0])) == 1.0
assert np.isclose(team3_ndcg_at_k(np.array([1.0, 2.0]), np.array([3, 0])), 1 / np.log2(3))

In [ ]:
TEAM3_RANKINGS = {"random order": None, "history popularity": "pop_history",
                  "same product, all history": "pooled_product"}
DRAWS = 20
frame = validation.sort_values("session_id", kind="stable")
bounds = np.r_[0, np.flatnonzero(np.diff(frame.session_id.to_numpy())) + 1, len(frame)]
labels_all = frame.label.to_numpy()
rows, sampled_parts = [], []
for lo, hi in zip(bounds[:-1], bounds[1:]):
    labels = labels_all[lo:hi]
    if not (labels > 0).any():
        continue  # Team 3 also drops lists that end up empty
    session_id = int(frame.session_id.iat[lo])
    keep = team3_sample(labels, np.random.default_rng([4222, 3, session_id]))
    sampled_parts.append(frame.iloc[lo:hi].iloc[keep])
    row = {"session_id": session_id, "n_full": hi - lo, "n_sampled": len(keep)}
    for name, feature in TEAM3_RANKINGS.items():
        scores = np.zeros(hi - lo) if feature is None else frame[feature].iloc[lo:hi].fillna(0).to_numpy(float)
        row[f"{name}|sampled"] = team3_scores(scores[keep], labels[keep], DRAWS, [4222, 4, session_id])
        row[f"{name}|full"] = team3_scores(scores, labels, DRAWS, [4222, 5, session_id])
    rows.append(row)
team3_runs = pd.DataFrame(rows).set_index("session_id")
sampled = pd.concat(sampled_parts, ignore_index=True)
print(f"searches {len(team3_runs):,}; mean list length full {team3_runs.n_full.mean():.1f}, "
      f"sampled {team3_runs.n_sampled.mean():.1f}; sampled lists with <= 10 candidates "
      f"{(team3_runs.n_sampled <= 10).mean():.1%}")

Four protocols side by side for each ranking: Team 3's protocol (downsampled lists, exponential gains), downsampled lists with our exact metric, full lists with Team 3's formula, and our protocol (full lists, linear gains, exact tie expectation).

In [ ]:
table = {}
for name, feature in TEAM3_RANKINGS.items():
    sampled_ndcg = team3_runs[f"{name}|sampled"].map(lambda x: x[0]).mean()
    sampled_hit = team3_runs[f"{name}|sampled"].map(lambda x: x[1]).mean()
    full_ndcg = team3_runs[f"{name}|full"].map(lambda x: x[0]).mean()
    sampled["score"] = 0.0 if feature is None else sampled[feature].fillna(0).astype(float)
    ours_sampled = evaluate(sampled, "score").ndcg.mean()
    table[name] = {"Team 3 protocol: NDCG@10": sampled_ndcg, "Team 3 protocol: HR@10": sampled_hit,
                   "downsampled, our metric": ours_sampled, "full lists, Team 3 formula": full_ndcg,
                   "full lists, our metric (E09)": summarise(per_search[name]).ndcg_at_10.iloc[0]}
team3_table = pd.DataFrame(table).T
team3_table.loc["Team 3 model (reported)"] = [0.568, 0.978, np.nan, np.nan, np.nan]
team3_table

## Summary of Team 3, part 1 (E10)

- **Their protocol makes random order look strong:** on our 19,772 validation searches, downsampling to all positives plus four negatives each (9.4 instead of 89.2 candidates; 80.8% of lists have at most 10) gives random order **NDCG@10 0.532 and HR@10 0.992**. Their model reported 0.568 and 0.978, only +0.036 NDCG above random and a lower hit rate than random.
- **Simple rules beat their reported model on their own protocol:** same product in history 0.585, history popularity 0.742.
- **Downsampling is the cause, not the gain formula:** on downsampled lists their exponential formula and our linear exact metric give almost the same values (random 0.532 vs 0.535); on full lists both give about 0.084.
- **Consequence:** their reported numbers cannot show a model's ranking quality. Any comparison with Team 3 must re-evaluate their model on complete lists with our protocol (part 2: correction ladder).
- Caveat: their reported numbers come from their own test split, not our validation searches; both are uniform samples of the same data.